<div style="background-color:#0e2233;background-image:linear-gradient(135deg,#0b1a29,#123049);color:#ffffff;padding:26px 28px 24px;border-radius:12px;border:1px solid #2a4a66;">
<div style="color:#a0d2ff;font-size:0.78em;font-weight:600;letter-spacing:0.18em;">AI &amp; ROBOTICS · NHÁNH AI ENGINEER · NOTEBOOK THỰC HÀNH</div>
<div style="width:56px;height:4px;background-color:#E07A1F;margin:14px 0 12px;"></div>
<h1 style="color:#ffffff;font-size:2.3em;line-height:1.15;margin:0;border:0;">Chương 21 · Feature engineering</h1>
<p style="color:#cfe6ff;font-size:1.12em;line-height:1.5;margin:10px 0 0;">Biến dữ liệu thô thành đặc trưng tốt: mã hóa one-hot, ColumnTransformer, log cho cột lệch, đặc trưng tương tác, và đo độ quan trọng bằng permutation.</p>
</div>

### Sau notebook này bạn sẽ

- Mã hóa cột phân loại đúng cách.
- Gom mọi bước xử lý cột vào ColumnTransformer trong Pipeline.
- Tạo đặc trưng tương tác khi mô hình tuyến tính cần.
- Đo độ quan trọng bằng permutation importance.

<div>
<span style="display:inline-block;margin:4px 6px 0 0;padding:4px 12px;border-radius:99px;background-color:#FDF1E6;color:#2b2b2b;border:1px solid #E07A1F;font-size:0.9em;"><b>Thời gian chạy:</b> dưới 1 phút</span>
<span style="display:inline-block;margin:4px 6px 0 0;padding:4px 12px;border-radius:99px;background-color:#FDF1E6;color:#2b2b2b;border:1px solid #E07A1F;font-size:0.9em;"><b>Runtime:</b> CPU là đủ, không cần GPU</span>
<span style="display:inline-block;margin:4px 6px 0 0;padding:4px 12px;border-radius:99px;background-color:#FDF1E6;color:#2b2b2b;border:1px solid #E07A1F;font-size:0.9em;"><b>Trang web của chương:</b> <a href="https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch21-feature-engineering/01-ma-hoa-phan-loai/" style="color:#1F5F8B;">mở bài đọc</a></span>
</div>

### Mục lục

1. [Trang 1 · Mã hóa đặc trưng phân loại](#trang-1)
2. [Trang 2 · ColumnTransformer và biến đổi cột số](#trang-2)
3. [Trang 3 · Tạo đặc trưng mới](#trang-3)
4. [Trang 4 · Đo độ quan trọng đúng cách](#trang-4)
5. [Trang 5 · Bài tập](#trang-5)

### Cách chạy

Bấm vào một ô code rồi nhấn **Shift + Enter** để chạy ô đó và nhảy sang ô kế tiếp. Muốn chạy hết một lượt thì chọn menu **Runtime > Run all**. Các ô nên chạy theo thứ tự từ trên xuống, vì ô sau dùng biến của ô trước.

<a name="trang-1" id="trang-1"></a>

## Trang 1 · Mã hóa đặc trưng phân loại

One-hot trên Bảng 4.1.

*Đọc kèm trên web: [Trang 1 · Mã hóa đặc trưng phân loại](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch21-feature-engineering/01-ma-hoa-phan-loai/)*

In [ ]:
import io                                                    # đọc chuỗi như file
import numpy as np                                           # tính toán
import pandas as pd                                          # bảng dữ liệu
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler   # mã hóa và chuẩn hóa
hang = ["ten,lop,gio,ngu,diem",                              # dòng tiêu đề
        "A,L1,1,7,2", "B,L1,2,6,4", "C,L1,3,8,5",            # Bảng 4.1
        "D,L1,4,7,7", "E,L2,5,6,6", "F,L2,2,,3.5",           # F thiếu giờ ngủ
        "G,L2,6,8,8.5", "H,L2,3,5,4", "I,L1,7,7,9",          # tiếp
        "K,L2,4,6,6.5"]                                      # bạn cuối
csv = "\n".join(hang)                                       # nối thành một chuỗi CSV
df = pd.read_csv(io.StringIO(csv))                           # đọc thành DataFrame
oh = OneHotEncoder(sparse_output=False).fit(df[["lop"]])     # one-hot cột lớp
print(oh.get_feature_names_out(), oh.transform(df[["lop"]])[:3])   # tên cột mới, 3 hàng đầu

Output mong đợi:

```text
['lop_L1' 'lop_L2'] [[1. 0.]
 [1. 0.]
 [1. 0.]]
```

<a name="trang-2" id="trang-2"></a>

## Trang 2 · ColumnTransformer và biến đổi cột số

ColumnTransformer, Pipeline, và log cho cột lệch.

*Đọc kèm trên web: [Trang 2 · ColumnTransformer và biến đổi cột số](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch21-feature-engineering/02-column-transformer/)*

In [ ]:
from sklearn.compose import ColumnTransformer               # xử lý từng nhóm cột
from sklearn.impute import SimpleImputer                    # điền giá trị thiếu
from sklearn.pipeline import make_pipeline                  # ghép bước
from sklearn.linear_model import LinearRegression, LogisticRegression   # mô hình
ct = ColumnTransformer([                                    # hai đường xử lý
    ("so", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), ["gio", "ngu"]),   # cột số
    ("chu", OneHotEncoder(), ["lop"]),                      # cột chữ
])                                                          # cột không liệt kê bị bỏ
Z = ct.fit_transform(df)                                    # học và biến đổi
print(Z.shape, Z[5].round(4), ct.get_feature_names_out())   # hàng của bạn F
m = make_pipeline(ct, LinearRegression()).fit(df, df["diem"])   # Pipeline nhận DataFrame thô
print(m[-1].coef_.round(4), round(m[-1].intercept_, 4))     # hệ số và hệ số chặn

Output mong đợi:

```text
(10, 4) [-0.9488  0.3333  0.      1.    ] ['so__gio' 'so__ngu' 'chu__lop_L1' 'chu__lop_L2']
[ 2.0287  0.1964  0.1242 -0.1242] 5.55
```

<div style="border-left:6px solid #2E8B57;background-color:#EAF6EF;color:#2b2b2b;padding:12px 18px 14px;border-radius:8px;margin:10px 0;">
<div style="color:#1E6B42;font-weight:700;font-size:0.85em;letter-spacing:0.08em;">ĐỌC KẾT QUẢ</div>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;">Khi đã biết giờ học, lớp L1 hay L2 chỉ chênh khoảng 0,25 điểm. Chênh lệch 0,3 điểm trung bình ở Chương 4 phần lớn là do L2 học nhiều giờ hơn.</p>
</div>

In [ ]:
rng = np.random.default_rng(0)                                         # seed
x = rng.lognormal(3, 1, 1000)                                          # 1000 giá trị lệch phải
lech = lambda v: float(((v - v.mean()) ** 3).mean() / v.std() ** 3)    # độ lệch (skewness)
print(round(float(np.mean(x)), 2), round(float(np.median(x)), 2), round(lech(x), 3), round(lech(np.log(x)), 3))   # trung bình, trung vị, độ lệch trước và sau log

Output mong đợi:

```text
30.63 18.64 3.875 -0.073
```

<a name="trang-3" id="trang-3"></a>

## Trang 3 · Tạo đặc trưng mới

Đặc trưng tương tác: XOR và bộ tiểu đường.

*Đọc kèm trên web: [Trang 3 · Tạo đặc trưng mới](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch21-feature-engineering/03-tao-dac-trung-moi/)*

In [ ]:
from sklearn.model_selection import cross_val_score, train_test_split   # cross-validation, chia
rng = np.random.default_rng(0)                          # seed
X = rng.uniform(-1, 1, (400, 2))                        # 400 điểm trong hình vuông
y = ((X[:, 0] * X[:, 1]) > 0).astype(int)               # cùng dấu thì 1
print(round(cross_val_score(LogisticRegression(), X, y, cv=5).mean(), 4))   # chỉ hai đặc trưng gốc
X2 = np.c_[X, X[:, 0] * X[:, 1]]                        # thêm cột tích
print(round(cross_val_score(LogisticRegression(), X2, y, cv=5).mean(), 4))  # có cột tích

Output mong đợi:

```text
0.4825
0.955
```

In [ ]:
from sklearn.datasets import load_diabetes                                   # bộ của Chương 9
Xd, yd = load_diabetes(return_X_y=True, as_frame=True, scaled=False)          # đơn vị gốc
Xdtr, Xdte, ydtr, ydte = train_test_split(Xd, yd, test_size=0.2, random_state=42)   # cách chia Chương 9
goc = cross_val_score(make_pipeline(StandardScaler(), LinearRegression()), Xdtr, ydtr, cv=5, scoring="r2").mean()   # trước
Xm = Xdtr.copy()                                                              # bản sao để thêm cột
Xm["bmi_x_bp"] = Xm["bmi"] * Xm["bp"]                                         # tương tác
Xm["bmi2"] = Xm["bmi"] ** 2                                                   # bình phương
moi = cross_val_score(make_pipeline(StandardScaler(), LinearRegression()), Xm, ydtr, cv=5, scoring="r2").mean()   # sau
print(round(goc, 4), round(moi, 4))                                           # không giúp gì

Output mong đợi:

```text
0.4493 0.4475
```

<div style="border-left:6px solid #E07A1F;background-color:#FDF1E6;color:#2b2b2b;padding:12px 18px 14px;border-radius:8px;margin:10px 0;">
<div style="color:#9A4F0E;font-weight:700;font-size:0.85em;letter-spacing:0.08em;">VÌ SAO LẠI KHÔNG TỐT HƠN?</div>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;">Hai giả thuyết (BMI và huyết áp cộng hưởng, BMI tăng nhanh dần) không đúng trên dữ liệu này. Hai cột mới chỉ thêm hai hệ số phải ước lượng.</p>
</div>

<a name="trang-4" id="trang-4"></a>

## Trang 4 · Đo độ quan trọng đúng cách

Impurity importance và permutation importance, với một cột nhiễu.

*Đọc kèm trên web: [Trang 4 · Đo độ quan trọng đúng cách](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch21-feature-engineering/04-do-quan-trong/)*

In [ ]:
from sklearn.ensemble import RandomForestRegressor                 # rừng
from sklearn.inspection import permutation_importance              # permutation importance
rf = RandomForestRegressor(300, random_state=0).fit(Xdtr, ydtr)    # 300 cây
pi = permutation_importance(rf, Xdte, ydte, n_repeats=20, random_state=0)   # xáo mỗi cột 20 lần
o = np.argsort(pi.importances_mean)[::-1][:4]                      # 4 cột quan trọng nhất
print([(c_, round(float(v), 4)) for c_, v in zip(Xd.columns[o], pi.importances_mean[o])])   # permutation
o2 = np.argsort(rf.feature_importances_)[::-1][:4]                 # theo impurity
print([(c_, round(float(v), 4)) for c_, v in zip(Xd.columns[o2], rf.feature_importances_[o2])])   # impurity

Output mong đợi:

```text
[('s5', 0.2323), ('bmi', 0.2127), ('bp', 0.0229), ('s6', 0.0177)]
[('bmi', 0.348), ('s5', 0.237), ('bp', 0.0929), ('s6', 0.0678)]
```

In [ ]:
rng = np.random.default_rng(0)                                     # seed
Xn, Xnt = Xdtr.copy(), Xdte.copy()                                 # bản sao train, test
Xn["nhieu"] = rng.normal(size=len(Xn))                             # cột ngẫu nhiên cho train
Xnt["nhieu"] = rng.normal(size=len(Xnt))                           # và cho test
rfn = RandomForestRegressor(300, random_state=0).fit(Xn, ydtr)     # rừng có cột nhiễu
pin = permutation_importance(rfn, Xnt, ydte, n_repeats=20, random_state=0)   # permutation
print(round(float(rfn.feature_importances_[-1]), 4), round(float(pin.importances_mean[-1]), 4))   # impurity và permutation của cột nhiễu
print(list(Xn.columns[np.argsort(rfn.feature_importances_)[::-1][:5]]))   # 5 cột đứng đầu theo impurity

Output mong đợi:

```text
0.0742 -0.0044
['bmi', 's5', 'bp', 'nhieu', 's6']
```

<div style="border-left:6px solid #C0392B;background-color:#FBECEA;color:#2b2b2b;padding:12px 18px 14px;border-radius:8px;margin:10px 0;">
<div style="color:#962D22;font-weight:700;font-size:0.85em;letter-spacing:0.08em;">LỖI HAY GẶP</div>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;"><b>Tin độ quan trọng impurity.</b> Cột nhiễu đứng hạng tư vì cây sâu luôn tìm được ngưỡng may mắn trên cột có nhiều giá trị. Permutation importance trên test cho nó gần 0.</p>
</div>

<a name="trang-5" id="trang-5"></a>

## Trang 5 · Bài tập

Ba bài tập. Làm ra giấy trước, rồi mới mở đáp án.

*Đọc kèm trên web: [Trang 5 · Bài tập](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch21-feature-engineering/05-bai-tap/)*

### Bài 21.1 · One-hot cho ba tỉnh

Viết tay ma trận one-hot cho HN, HCM, DN, HCM, HN.
<details><summary>Xem đáp án</summary>

Ba cột xếp theo chữ cái: DN, HCM, HN. Hàng đầu (HN) là $(0, 0, 1)$. Code ở ô ngay dưới.
</details>

In [ ]:
t = pd.DataFrame({"tinh": ["HN", "HCM", "DN", "HCM", "HN"]})          # năm hàng
oh3 = OneHotEncoder(sparse_output=False).fit(t)                      # one-hot
print(oh3.get_feature_names_out(), oh3.transform(t).astype(int).tolist())   # tên cột và ma trận

Output mong đợi:

```text
['tinh_DN' 'tinh_HCM' 'tinh_HN'] [[0, 0, 1], [0, 1, 0], [1, 0, 0], [0, 1, 0], [0, 0, 1]]
```

### Bài 21.2 · Gán số thứ tự sai

Ba lớp có điểm trung bình 5, 8, 4. So dự đoán của hồi quy với mã hóa thứ tự và one-hot.
<details><summary>Xem đáp án</summary>

Thứ tự: 6,17; 5,67; 5,17, sai cả ba, lớp giỏi nhất bị đoán gần kém nhất. One-hot: đúng 5, 8, 4. Code ở ô ngay dưới.
</details>

In [ ]:
lop = pd.DataFrame({"lop": ["L1"] * 4 + ["L2"] * 4 + ["L3"] * 4})       # ba lớp
yl = np.array([5] * 4 + [8] * 4 + [4] * 4, dtype=float)                  # điểm
r1 = LinearRegression().fit(OrdinalEncoder().fit_transform(lop), yl)     # mã hóa thứ tự
print(r1.predict([[0], [1], [2]]).round(4))                              # dự đoán ba lớp
r2 = LinearRegression().fit(OneHotEncoder(sparse_output=False).fit_transform(lop), yl)   # one-hot
print(r2.predict(np.eye(3)).round(4))                                    # dự đoán ba lớp

Output mong đợi:

```text
[6.1667 5.6667 5.1667]
[5. 8. 4.]
```

### Bài 21.3 · Permutation importance bằng tay

Xáo một lần cột `bmi` của tập test, so $R^2$ của hồi quy tuyến tính trước và sau.
<details><summary>Xem đáp án</summary>

Từ 0,4526 xuống 0,1842: giảm 0,27. Code ở ô ngay dưới.
</details>

In [ ]:
from sklearn.metrics import r2_score                                     # R²
ml = LinearRegression().fit(Xdtr, ydtr)                                  # hồi quy tuyến tính
g = r2_score(ydte, ml.predict(Xdte))                                     # R² gốc
Xx = Xdte.copy()                                                         # bản sao test
Xx["bmi"] = np.random.default_rng(0).permutation(Xx["bmi"].values)       # xáo riêng cột bmi
print(round(g, 4), round(r2_score(ydte, ml.predict(Xx)), 4), round(g - r2_score(ydte, ml.predict(Xx)), 4))   # trước, sau, độ tụt

Output mong đợi:

```text
0.4526 0.1842 0.2684
```